# The AI Economy — Interactive Explorer
### Companion to *Governing the AI Economic Transition* (March 2026)

This notebook lets you explore the parametric model behind the policy brief. You can:

- **Compare pre-defined scenarios** from the paper
- **Build your own scenario** using sliders for the key parameters
- **Probe specific mechanisms** — ownership structure, enforcement gaps, governance decay

No coding required. Run each cell with **Shift+Enter**, then adjust sliders to explore.

---
*Model calibrated to a US-like economy ($28T nominal GDP, 160M workers, 2025 baseline). All parameters are explicit and adjustable.*

In [ ]:
import sys, os
os.chdir(os.path.dirname(os.path.abspath("explorer.ipynb")) if "__file__" not in dir() else os.path.dirname(__file__))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, HTML
import warnings
warnings.filterwarnings("ignore")

# Import the model
sys.path.insert(0, ".")
from model import run, SCENARIOS

# Colour palette for scenarios
PALETTE = [
    "#e74c3c", "#e67e22", "#f1c40f", "#2ecc71",
    "#1abc9c", "#3498db", "#9b59b6", "#34495e",
    "#e91e63", "#00bcd4", "#ff5722", "#607d8b",
]

print("✓ Model loaded.")
print(f"  {len(SCENARIOS)} pre-defined scenarios available.")
print("  Run cells below with Shift+Enter to explore.")


---
## Part 1 — Compare pre-defined scenarios

Select any combination of the paper's scenarios and compare them across six key metrics.


In [ ]:
CORE_SCENARIOS = [
    "Fast / No yeomen / Low tax",
    "Fast / No yeomen / High tax",
    "Fast / Moderate yeomen / Med tax",
    "Fast / High yeomen / High tax",
    "Fast / Public AI 90%",
    "Fast / High yeomen / Tax reform",
    "Fast / High yeomen / No tax reform",
    "Fast / High yeomen / Firm rebate",
]

scenario_select = widgets.SelectMultiple(
    options=list(SCENARIOS.keys()),
    value=CORE_SCENARIOS[:5],
    rows=12,
    description="Scenarios",
    layout=widgets.Layout(width="480px"),
    style={"description_width": "80px"},
)

metric_select = widgets.ToggleButtons(
    options=[
        ("Nominal GDP", "nom_gdp_bn"),
        ("Real GDP index", "gdp_real_idx"),
        ("Gini", "gini_proxy"),
        ("Welfare (eff.)", "effective_welfare_k"),
        ("Fiscal space", "fiscal_space_bn"),
        ("Interest % rev", "interest_pct_revenue"),
    ],
    description="Metric:",
    style={"description_width": "60px", "button_width": "120px"},
)

compare_btn = widgets.Button(description="Compare selected", button_style="primary",
                              layout=widgets.Layout(width="180px"))
compare_out = widgets.Output()

def do_compare(_):
    compare_out.clear_output(wait=True)
    selected = list(scenario_select.value)
    if not selected:
        with compare_out:
            print("Select at least one scenario.")
        return
    col = metric_select.value
    label_map = {v: k for k, v in dict(metric_select.options).items()}
    label = label_map[col]

    fig, ax = plt.subplots(figsize=(11, 5))
    for i, name in enumerate(selected):
        p = SCENARIOS[name]
        df = run(**p)
        scale = 1/1000 if col in ("nom_gdp_bn", "fiscal_space_bn") else 1
        unit  = "T" if col in ("nom_gdp_bn", "fiscal_space_bn") else (
                "k/yr" if col == "effective_welfare_k" else (
                "%" if col in ("gini_proxy", "interest_pct_revenue") else ""))
        ax.plot(df.year, df[col] * scale, label=name,
                color=PALETTE[i % len(PALETTE)], linewidth=2)

    ax.set_title(f"{label} — scenario comparison", fontsize=13)
    ax.set_xlabel("Year")
    ylabel = f"${label} ({unit})" if "$" not in label else f"{label} ({unit})"
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8, loc="upper left")
    ax.axvline(2025, color="#ccc", linewidth=1)
    fig.tight_layout()
    with compare_out:
        plt.show()

compare_btn.on_click(do_compare)

ui = widgets.VBox([
    widgets.HBox([scenario_select,
                  widgets.VBox([metric_select, compare_btn])]),
    compare_out,
])
display(ui)


---
## Part 2 — Build your own scenario

Use the sliders to construct a custom scenario. The dashboard updates when you click **Run scenario**.


In [ ]:
slider_style = {"description_width": "220px"}
slider_layout = widgets.Layout(width="520px")

w_mid = widgets.IntSlider(
    value=5, min=2, max=18, step=1, description="Automation speed (midpoint yrs)",
    style=slider_style, layout=slider_layout,
    continuous_update=False,
)
w_mid_label = widgets.HTML("<small>Fast=5 yrs · Medium=9 · Slow=14 · Very slow=18</small>")

w_yeomen = widgets.FloatSlider(
    value=0.0, min=0.0, max=0.70, step=0.05,
    description="Yeomen capital fraction",
    style=slider_style, layout=slider_layout, continuous_update=False,
    readout_format=".0%",
)
w_dao = widgets.FloatSlider(
    value=0.0, min=0.0, max=0.30, step=0.05,
    description="DAO / commons fraction",
    style=slider_style, layout=slider_layout, continuous_update=False,
    readout_format=".0%",
)
w_public_ai = widgets.FloatSlider(
    value=0.0, min=0.0, max=0.90, step=0.10,
    description="Public AI capital fraction",
    style=slider_style, layout=slider_layout, continuous_update=False,
    readout_format=".0%",
)
w_taxk = widgets.FloatSlider(
    value=0.20, min=0.10, max=0.60, step=0.05,
    description="Capital tax rate",
    style=slider_style, layout=slider_layout, continuous_update=False,
    readout_format=".0%",
)
w_enf = widgets.FloatSlider(
    value=1.0, min=0.20, max=1.0, step=0.05,
    description="Enforcement (1.0=large bloc)",
    style=slider_style, layout=slider_layout, continuous_update=False,
    readout_format=".0%",
)
w_friction = widgets.FloatSlider(
    value=0.18, min=0.0, max=0.25, step=0.01,
    description="Yeomen tax friction (0=reformed)",
    style=slider_style, layout=slider_layout, continuous_update=False,
    readout_format=".0%",
)
w_rebate = widgets.FloatSlider(
    value=0.0, min=0.0, max=0.25, step=0.05,
    description="Firm yeomen rebate rate",
    style=slider_style, layout=slider_layout, continuous_update=False,
    readout_format=".0%",
)
w_levy = widgets.FloatSlider(
    value=0.0, min=0.0, max=1.0, step=0.1,
    description="Progressive compute levy",
    style=slider_style, layout=slider_layout, continuous_update=False,
    readout_format=".1f",
)

run_btn = widgets.Button(description="▶  Run scenario", button_style="success",
                          layout=widgets.Layout(width="160px", height="40px"))
custom_out = widgets.Output()

def run_custom(_):
    custom_out.clear_output(wait=True)
    df = run(
        mid=w_mid.value,
        yeomen=w_yeomen.value,
        dao_frac=w_dao.value,
        public_ai_frac=w_public_ai.value,
        tax_k=w_taxk.value,
        enforcement=w_enf.value,
        yeomen_tax_friction=w_friction.value,
        firm_yeomen_rebate=w_rebate.value,
        levy_prog=w_levy.value,
    )

    fig = plt.figure(figsize=(14, 8))
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)

    def ax_line(pos, col, title, scale=1, unit="", color="#2c3e50", ref_col=None, ref_label=None):
        ax = fig.add_subplot(pos)
        ax.plot(df.year, df[col] * scale, color=color, linewidth=2.5)
        if ref_col:
            ref_df = run(**SCENARIOS["Fast / No yeomen / Low tax"])
            ax.plot(ref_df.year, ref_df[ref_col] * scale, color="#ccc",
                    linewidth=1.5, linestyle="--", label=ref_label or "Baseline")
            ax.legend(fontsize=7)
        ax.set_title(title, fontsize=11)
        ax.set_xlabel("Year", fontsize=9)
        ax.set_ylabel(unit, fontsize=9)
        ax.axvline(2025, color="#eee", linewidth=1)
        return ax

    ax_line(gs[0, 0], "nom_gdp_bn", "Nominal GDP ($T)", 1/1000, "$T",
            "#3498db", "nom_gdp_bn", "Concentrated low tax")
    ax_line(gs[0, 1], "gini_proxy", "Gini (market income)", unit="Gini",
            color="#e74c3c", ref_col="gini_proxy")
    ax_line(gs[0, 2], "effective_welfare_k", "Effective welfare (k/yr displaced worker)",
            unit="$k/yr", color="#27ae60", ref_col="effective_welfare_k")
    ax_line(gs[1, 0], "fiscal_space_bn", "Fiscal space ($T)", 1/1000, "$T",
            color="#8e44ad", ref_col="fiscal_space_bn")
    ax_line(gs[1, 1], "labor_share_pct", "Labour share of income (%)",
            unit="%", color="#e67e22", ref_col="labor_share_pct")
    ax_line(gs[1, 2], "yeomen_income_bn", "Yeomen income ($T)", 1/1000, "$T",
            color="#1abc9c")

    # Snapshot table
    t10 = df[df.year == 2035].iloc[0]
    t20 = df[df.year == 2045].iloc[0]
    snap = (
        f"  t+10 (2035): GDP ${t10.nom_gdp_bn/1000:.1f}T | Gini {t10.gini_proxy:.3f} | "
        f"Welfare ${t10.effective_welfare_k:.0f}k | Fiscal space ${t10.fiscal_space_bn/1000:.1f}T\n"
        f"  t+20 (2045): GDP ${t20.nom_gdp_bn/1000:.1f}T | Gini {t20.gini_proxy:.3f} | "
        f"Welfare ${t20.effective_welfare_k:.0f}k | Fiscal space ${t20.fiscal_space_bn/1000:.1f}T"
    )
    fig.suptitle("Custom scenario  (dashed = concentrated / low tax baseline)", fontsize=12)
    fig.text(0.5, 0.01, snap, ha="center", fontsize=9, family="monospace",
             bbox=dict(boxstyle="round", facecolor="#f8f8f8", alpha=0.8))
    with custom_out:
        plt.show()

run_btn.on_click(run_custom)

controls = widgets.VBox([
    widgets.HTML("<b>Ownership structure</b>"),
    w_yeomen, w_dao, w_public_ai,
    widgets.HTML("<b>Tax policy</b>"),
    w_taxk, w_enf, w_levy,
    widgets.HTML("<b>Yeomen fiscal reform</b>"),
    w_friction, w_rebate,
    widgets.HTML("<b>Automation speed</b>"),
    w_mid, w_mid_label,
    run_btn,
])
display(widgets.VBox([controls, custom_out]))


---
## Part 3 — Probe specific mechanisms

Three targeted probes of the model's key structural findings.


In [ ]:
# ── Probe A: Ownership structure sweep ─────────────────────────────────────────
# Hold everything constant, vary yeomen fraction from 0% to 70%

print("Probe A: Sweeping yeomen fraction (0% → 70%), fast automation, 40% capital tax")
print("Running 8 scenarios...", end=" ")

yeomen_range = np.arange(0.0, 0.75, 0.10)
results_a = []
for yf in yeomen_range:
    df = run(mid=5, yeomen=yf, tax_k=0.40, enforcement=1.0, yeomen_tax_friction=0.18)
    t10 = df[df.year == 2035].iloc[0]
    t20 = df[df.year == 2045].iloc[0]
    results_a.append({
        "yeomen_frac": yf,
        "gini_t10": t10.gini_proxy,
        "welfare_t10": t10.effective_welfare_k,
        "fiscal_t10": t10.fiscal_space_bn / 1000,
        "gini_t20": t20.gini_proxy,
        "welfare_t20": t20.effective_welfare_k,
    })

print("done.")
ra = pd.DataFrame(results_a)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].plot(ra.yeomen_frac * 100, ra.gini_t10, "o-", color="#e74c3c", label="t+10")
axes[0].plot(ra.yeomen_frac * 100, ra.gini_t20, "s--", color="#e74c3c", alpha=0.6, label="t+20")
axes[0].axhline(0.41, color="#999", linewidth=1, linestyle=":", label="US today (~0.41)")
axes[0].set_title("Gini vs Yeomen fraction")
axes[0].set_xlabel("Yeomen fraction (%)")
axes[0].legend(fontsize=8)

axes[1].plot(ra.yeomen_frac * 100, ra.welfare_t10, "o-", color="#27ae60", label="t+10")
axes[1].plot(ra.yeomen_frac * 100, ra.welfare_t20, "s--", color="#27ae60", alpha=0.6, label="t+20")
axes[1].set_title("Effective welfare ($k/yr) vs Yeomen fraction")
axes[1].set_xlabel("Yeomen fraction (%)")
axes[1].set_ylabel("$k/yr (displaced worker)")
axes[1].legend(fontsize=8)

axes[2].plot(ra.yeomen_frac * 100, ra.fiscal_t10, "o-", color="#8e44ad", label="t+10")
axes[2].axhline(0, color="#ccc", linewidth=1)
axes[2].set_title("Fiscal space ($T) vs Yeomen fraction")
axes[2].set_xlabel("Yeomen fraction (%)")
axes[2].set_ylabel("$T")
axes[2].legend(fontsize=8)

fig.suptitle("Probe A: Ownership structure sweep  (fast automation, 40% capital tax)", fontsize=12)
fig.tight_layout()
plt.show()


In [ ]:
# ── Probe B: Enforcement gap — tax rate vs enforcement surface ──────────────────
# Shows how effective revenue varies across (statutory rate, enforcement) combinations

print("Probe B: Enforcement gap surface — statutory rate vs enforcement multiplier")
print("Running 30 scenarios...", end=" ")

tax_rates = [0.20, 0.30, 0.40, 0.50, 0.60]
enforcements = [0.30, 0.50, 0.70, 0.85, 1.00, 1.00]  # 6 points
enf_vals = [0.30, 0.50, 0.70, 0.85, 1.00]

surface = np.zeros((len(tax_rates), len(enf_vals)))
for i, tk in enumerate(tax_rates):
    for j, enf in enumerate(enf_vals):
        df = run(mid=5, yeomen=0.0, tax_k=tk, enforcement=enf)
        t10 = df[df.year == 2035].iloc[0]
        surface[i, j] = t10.fiscal_space_bn / 1000

print("done.")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap
im = ax1.imshow(surface, aspect="auto", cmap="RdYlGn", vmin=-2, vmax=8)
ax1.set_xticks(range(len(enf_vals)))
ax1.set_xticklabels([f"{e:.0%}" for e in enf_vals])
ax1.set_yticks(range(len(tax_rates)))
ax1.set_yticklabels([f"{r:.0%}" for r in tax_rates])
ax1.set_xlabel("Enforcement multiplier")
ax1.set_ylabel("Statutory capital tax rate")
ax1.set_title("Fiscal space at t+10 ($T)")
for i in range(len(tax_rates)):
    for j in range(len(enf_vals)):
        ax1.text(j, i, f"${surface[i,j]:.1f}T", ha="center", va="center", fontsize=8,
                 color="white" if surface[i,j] < 1 else "black")
plt.colorbar(im, ax=ax1, label="$T")

# Line chart: each enforcement level across tax rates
for j, enf in enumerate(enf_vals):
    ax2.plot([r * 100 for r in tax_rates], surface[:, j],
             "o-", label=f"Enforcement {enf:.0%}", color=PALETTE[j])
ax2.axhline(0, color="#ccc", linewidth=1)
ax2.set_xlabel("Statutory capital tax rate (%)")
ax2.set_ylabel("Fiscal space at t+10 ($T)")
ax2.set_title("Revenue vs rate at different enforcement levels")
ax2.legend(fontsize=8)

fig.suptitle("Probe B: Enforcement gap  (fast automation, no yeomen)", fontsize=12)
fig.tight_layout()
plt.show()

print("\nKey takeaway: A 60% rate at 30% enforcement (small open economy) often collects")
print("less than a 20% rate at 100% enforcement (large bloc with consumer-market leverage).")


In [ ]:
# ── Probe C: Yeomen fiscal friction — cost of current tax code ──────────────────
# Shows income impact of tax friction vs firm rebate policies

print("Probe C: Yeomen tax friction and firm rebate effects")
print("Running 12 scenarios...", end=" ")

friction_vals = np.arange(0.0, 0.22, 0.03)
rebate_vals = [0.0, 0.10, 0.20]

results_c = []
for fr in friction_vals:
    for rb in rebate_vals:
        df = run(mid=5, yeomen=0.60, tax_k=0.45, enforcement=1.0,
                 yeomen_tax_friction=fr, firm_yeomen_rebate=rb)
        t10 = df[df.year == 2035].iloc[0]
        results_c.append({
            "friction": fr,
            "rebate": rb,
            "welfare_t10": t10.effective_welfare_k,
            "gini_t10": t10.gini_proxy,
            "friction_cost_bn": t10.yeomen_friction_cost_bn,
        })

print("done.")
rc = pd.DataFrame(results_c)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

colors_c = ["#27ae60", "#e67e22", "#e74c3c"]
for k, rb in enumerate(rebate_vals):
    sub = rc[rc.rebate == rb]
    axes[0].plot(sub.friction * 100, sub.welfare_t10, "o-", color=colors_c[k],
                 label=f"Firm rebate {rb:.0%}")
    axes[1].plot(sub.friction * 100, sub.gini_t10, "o-", color=colors_c[k],
                 label=f"Firm rebate {rb:.0%}")
    axes[2].plot(sub.friction * 100, sub.friction_cost_bn / 1000, "o-", color=colors_c[k],
                 label=f"Firm rebate {rb:.0%}")

axes[0].set_title("Effective welfare t+10 ($k/yr)")
axes[0].set_xlabel("Tax friction rate (%)")
axes[0].set_ylabel("$k/yr")
axes[0].legend(fontsize=8)
axes[0].axvline(18, color="#ccc", linewidth=1, linestyle="--", label="Current code (~18%)")

axes[1].set_title("Gini t+10 (market income)")
axes[1].set_xlabel("Tax friction rate (%)")
axes[1].legend(fontsize=8)
axes[1].axvline(18, color="#ccc", linewidth=1, linestyle="--")

axes[2].set_title("Yearly friction cost ($T lost to yeomen)")
axes[2].set_xlabel("Tax friction rate (%)")
axes[2].set_ylabel("$T/yr")
axes[2].legend(fontsize=8)
axes[2].axvline(18, color="#ccc", linewidth=1, linestyle="--")

fig.suptitle("Probe C: Yeomen tax friction  (60% yeomen, fast automation, 45% capital tax)\n"
             "Dashed line = current US tax code (~18% discount on sole operator income)",
             fontsize=11)
fig.tight_layout()
plt.show()

# Print summary at current code friction level
base = rc[(rc.friction.round(2) == 0.18) & (rc.rebate == 0.0)].iloc[0]
reformed = rc[(rc.friction.round(2) == 0.0) & (rc.rebate == 0.0)].iloc[0]
rebated = rc[(rc.friction.round(2) == 0.18) & (rc.rebate == 0.20)].iloc[0]

print(f"\nAt 60% yeomen fraction, fast automation, 45% capital tax:")
print(f"  Current code (18% friction, no rebate):  welfare ${base.welfare_t10:.0f}k/yr | "
      f"friction cost ${base.friction_cost_bn/1000:.1f}T/yr")
print(f"  Reformed (0% friction):                  welfare ${reformed.welfare_t10:.0f}k/yr | "
      f"friction cost $0T/yr")
print(f"  Firm rebate 20% (18% friction):          welfare ${rebated.welfare_t10:.0f}k/yr | "
      f"friction cost ${rebated.friction_cost_bn/1000:.1f}T/yr")
